<div class="alert alert-info">
    <b>Variational Autoencoder (VAE) für Synthetische Datenerzeugung</b><br>
    Implementierung eines Variational Autoencoders zur Generation synthetischer hydrogeochemischer Daten.<br>
    Beinhaltet Analyse und Imputation fehlender Werte.
</div>

In [ ]:
# ------------------------- Installation externer Bibliotheken -------------------------
!pip install torch torchvision torchaudio scikit-learn pandas numpy matplotlib seaborn

In [ ]:
# ------------------------- Import notwendiger Bibliotheken -------------------------
import os
import glob
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
import matplotlib.pyplot as plt
import seaborn as sns

# ------------------------- Globale Plot-Einstellungen -------------------------
sns.set(style="whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

<div class="alert alert-info">
    <b>1. Daten Laden</b><br>
    Automatisches Laden der neuesten `Preprocessed_SOM_Ready.csv` aus dem Preprocessing-Verzeichnis.
</div>

In [ ]:
# ------------------------- Pfad definition und Laden der Daten -------------------------
current_dir = os.getcwd()
notebooks_dir = os.path.dirname(os.path.dirname(current_dir))
preprocessing_dir = os.path.join(notebooks_dir, '3.1_Preprocessing', 'Preprocessing')

if not os.path.exists(preprocessing_dir):
    preprocessing_dir = "../../3.1_Preprocessing/Preprocessing"

try:
    # ------------------------- Suche nach neuestem Preprocessing-Ordner -------------------------
    all_subdirs = [os.path.join(preprocessing_dir, d) for d in os.listdir(preprocessing_dir) if os.path.isdir(os.path.join(preprocessing_dir, d))]
    if not all_subdirs:
        raise FileNotFoundError("Keine Preprocessing-Ordner gefunden.")
        
    latest_subdir = max(all_subdirs, key=os.path.getmtime)
    data_path = os.path.join(latest_subdir, 'Preprocessed_SOM_Ready.csv')

    print(f"Lade Daten aus: {data_path}")

    # ------------------------- Einlesen der CSV-Datei -------------------------
    df = pd.read_csv(data_path)
    display(df.head())
    print(f"Datensatzgröße: {df.shape}")

except Exception as e:
    print(f"Fehler beim Laden der Daten: {e}")
    print("Bitte Pfad manuell prüfen.")

<div class="alert alert-info">
    <b>2. Analyse fehlender Werte</b><br>
    Visualisierung des Anteils fehlender Werte pro Variable.
</div>

In [ ]:
# ------------------------- Berechnung des Fehlanteils pro Feature -------------------------
missing_percentage = df.isna().mean() * 100

# ------------------------- Visualisierung als Barplot -------------------------
plt.figure(figsize=(12, 6))
sns.barplot(x=missing_percentage.index, y=missing_percentage.values, palette='viridis')
plt.xticks(rotation=90)
plt.title("Prozentsatz fehlender Werte pro Feature")
plt.ylabel("Fehlend (%)")
plt.axhline(95, color='r', linestyle='--', label='95% Schwelle')
plt.legend()
plt.tight_layout()
plt.show()

# ------------------------- Ausgabe der Top 10 Features mit fehlenden Werten -------------------------
print("Top 10 Features mit den meisten fehlenden Werten:")
print(missing_percentage.sort_values(ascending=False).head(10))

<div class="alert alert-info">
    <b>3. Datenvorbereitung und Imputation</b><br>
    1. Entfernen von Spalten mit > 95% fehlenden Werten.<br>
    2. Ausschluss von Metadaten.<br>
    3. Mean-Imputation für verbleibende Lücken.<br>
    4. Standardisierung der Daten.
</div>

In [ ]:
# ------------------------- Filterung leerer Spalten (>95% Missing) -------------------------
drop_threshold = 95.0
cols_to_drop = missing_percentage[missing_percentage > drop_threshold].index

if len(cols_to_drop) > 0:
    print(f"\nEntferne Spalten mit >{drop_threshold}% fehlenden Werten: {list(cols_to_drop)}")
    df_clean = df.drop(columns=cols_to_drop)
else:
    df_clean = df
    print("\nKeine Spalten entfernt.")

# ------------------------- Ausschluss von Metadaten -------------------------
cols_to_exclude = ['Database_number', 'database_name', 'Database_name', 'Date', 'Day', 'Month', 'Year']
cols_to_exclude = [c for c in cols_to_exclude if c in df_clean.columns]

if cols_to_exclude:
    print(f"\nEntferne explizit Metadaten-Spalten: {cols_to_exclude}")
    df_clean = df_clean.drop(columns=cols_to_exclude)

# ------------------------- Selektion numerischer Spalten -------------------------
numeric_cols = df_clean.select_dtypes(include=[np.number]).columns
data_values = df_clean[numeric_cols].values

# ------------------------- Imputation fehlender Werte (Mean) -------------------------
print("Führe Mean-Imputation für verbleibende NaN-Werte durch...")
imputer = SimpleImputer(strategy='mean')
data_imputed = imputer.fit_transform(data_values)

# ------------------------- Skalierung (StandardScaler) -------------------------
scaler = StandardScaler()
data_scaled = scaler.fit_transform(data_imputed)

# ------------------------- Überprüfung auf NaNs -------------------------
if np.isnan(data_scaled).sum() == 0:
    print("Erfolg: Keine NaNs mehr im Datensatz vorhanden.")
else:
    print("WARNUNG: Immer noch NaNs vorhanden!")

# ------------------------- Erstellung PyTorch Tensoren und DataLoader -------------------------
tensor_data = torch.FloatTensor(data_scaled)

batch_size = 64
dataset = TensorDataset(tensor_data)
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

print(f"Verwendete Features ({len(numeric_cols)}): {list(numeric_cols)}")

<div class="alert alert-info">
    <b>4. Definition der VAE-Modellarchitektur</b>
</div>

In [ ]:
class VAE(nn.Module):
    def __init__(self, input_dim, hidden_dim=64, latent_dim=10):
        super(VAE, self).__init__()
        
        # ------------------------- Encoder Architektur -------------------------
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.fc2_mu = nn.Linear(hidden_dim, latent_dim)      # Mittelwert
        self.fc2_logvar = nn.Linear(hidden_dim, latent_dim)  # Log Varianz
        
        # ------------------------- Decoder Architektur -------------------------
        self.fc3 = nn.Linear(latent_dim, hidden_dim)
        self.fc4 = nn.Linear(hidden_dim, input_dim)
        
        self.activation = nn.ReLU()
        
    def encode(self, x):
        # ------------------------- Forward Pass Encoder -------------------------
        h1 = self.activation(self.fc1(x))
        return self.fc2_mu(h1), self.fc2_logvar(h1)
    
    def reparameterize(self, mu, logvar):
        # ------------------------- Reparameterisierungstrick -------------------------
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std
    
    def decode(self, z):
        # ------------------------- Forward Pass Decoder -------------------------
        h3 = self.activation(self.fc3(z))
        return self.fc4(h3) 
    
    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        return self.decode(z), mu, logvar

# ------------------------- Modell-Initialisierung -------------------------
input_dim = data_scaled.shape[1]
latent_dim = 8
model = VAE(input_dim, hidden_dim=64, latent_dim=latent_dim)
optimizer = optim.Adam(model.parameters(), lr=1e-3)

<div class="alert alert-info">
    <b>5. Training des Modells</b>
</div>

In [ ]:
def loss_function(recon_x, x, mu, logvar):
    # ------------------------- Berechnung Loss (MSE + KLD) -------------------------
    MSE = nn.MSELoss(reduction='sum')(recon_x, x)
    KLD = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
    return MSE + KLD

epochs = 100
loss_history = []

# ------------------------- Trainingsschleife -------------------------
model.train()
for epoch in range(epochs):
    train_loss = 0
    for batch_idx, (data_batch,) in enumerate(dataloader):
        optimizer.zero_grad()
        recon_batch, mu, logvar = model(data_batch)
        loss = loss_function(recon_batch, data_batch, mu, logvar)
        loss.backward()
        train_loss += loss.item()
        optimizer.step()
    
    avg_loss = train_loss / len(dataloader.dataset)
    loss_history.append(avg_loss)
    
    if epoch % 10 == 0:
        print(f'Epoch {epoch}: Average Loss: {avg_loss:.4f}')

# ------------------------- Visualisierung Lernkurve -------------------------
plt.plot(loss_history)
plt.title('Training Loss Curve')
plt.xlabel('Epoch')
plt.ylabel('Loss (MSE + KLD)')
plt.show()

<div class="alert alert-info">
    <b>6. Generierung und Validierung synthetischer Daten</b>
</div>

In [ ]:
# ------------------------- Generierung synthetischer Daten -------------------------
model.eval()
with torch.no_grad():
    n_samples = 1000
    z = torch.randn(n_samples, latent_dim)
    generated_data_scaled = model.decode(z).numpy()
    
    # ------------------------- Rücktransformation der Daten -------------------------
    generated_data = scaler.inverse_transform(generated_data_scaled)
    
    synthetic_df = pd.DataFrame(generated_data, columns=numeric_cols)

# ------------------------- Speichern der CSV -------------------------
output_file = "Synthetic_Hydrogeochemistry_Data_Imputed.csv"
synthetic_df.to_csv(output_file, index=False)
print(f"Gespeichert unter: {output_file}")

# ------------------------- Visueller Abgleich (KDE Plot) -------------------------
plot_cols = numeric_cols[:4]

plt.figure(figsize=(15, 10))
for i, col in enumerate(plot_cols):
    plt.subplot(2, 2, i+1)
    sns.kdeplot(df_clean[col], label='Original (Imputed)', fill=True, alpha=0.3)
    sns.kdeplot(synthetic_df[col], label='Synthetisch', fill=True, color='orange', alpha=0.3)
    plt.title(f"Verteilung: {col}")
    plt.legend()

plt.tight_layout()
plt.show()